In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from skimage.feature import hog
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

---------------------------------------------------------------------------------------------- Dataset Understanding and extract features
 ---------------------------------------------------------------------------------------------

In [2]:
base_path = r"D:\hady\3nd year\Second_term\Supervised_Learning\project\GTSRB"
train_csv = os.path.join(base_path, "Train.csv")
test_csv  = os.path.join(base_path, "Test.csv")
target_size = (32, 32)

train_data = pd.read_csv(train_csv)
test_data  = pd.read_csv(test_csv)

def load_and_extract_features(csv_file):
    data = pd.read_csv(csv_file)
    features_list = []
    labels_list = []
    images = []
    labels = []
    for idx, row in data.iterrows():
        img_path = os.path.join(base_path, row['Path'])
        class_id = row['ClassId']
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, target_size)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) 
            features = hog(gray, orientations=9, pixels_per_cell=(8,8), 
                           cells_per_block=(2,2), block_norm='L2-Hys')
            features_list.append(features)
            labels_list.append(class_id)
            images.append(img)
            labels.append(class_id)
    features_array = np.array(features_list, dtype='float32')
    labels_array = np.array(labels_list, dtype='int')
    images = np.array(images, dtype='float32') / 255.0  # normalize 0-1
    labels = np.array(labels)
    return features_array, labels_array,images, labels


X_train_feat, y_train_feat,X_train, y_train = load_and_extract_features(train_csv)
X_test_feat, y_test_feat,X_test, y_test   = load_and_extract_features(test_csv)


print("✅ Training set:")
print("Number of images:", X_train_feat.shape[0])
print("Number of classes:", len(np.unique(y_train_feat)))
print("Example labels:", np.unique(y_train_feat))
print("Sample structure (first 5 rows of CSV):")
print(train_data.head())

print("\n✅ Test set:")
print("Number of images:", X_test_feat.shape[0])
print("Number of classes:", len(np.unique(y_test_feat)))
print("Example labels:", np.unique(y_test_feat))
print("Sample structure (first 5 rows of CSV):")
print(test_data.head())

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\hady\\3nd year\\Second_term\\Supervised_Learning\\project\\GTSRB\\Train.csv'

---------------------------------------------------------------------------------------------- Exploratory Data Analysis (EDA)
 ---------------------------------------------------------------------------------------------

In [2]:
def show_samples(x,y,num_image = 5):
    plt.figure(figsize=(15, 5))
    classes_shown = 0
    seen_classes = set()
    
    for img, label in zip(x, y):
        if label not in seen_classes:
            plt.subplot(2, 5, classes_shown+1)
            plt.imshow(img)
            plt.title(f"Class {label}")
            plt.axis('off')
            seen_classes.add(label)
            classes_shown += 1
        if classes_shown >= num_image:
            break
    plt.suptitle("Sample images from different classes")
    plt.show()

show_samples(X_train, y_train, num_image=10)

NameError: name 'X_train' is not defined

In [ ]:
train_classes, train_counts = np.unique(y_train_feat, return_counts=True)
plt.figure(figsize=(12,5))
sns.barplot(x=train_classes, y=train_counts, palette="viridis" , hue=train_classes)
plt.title("Number of images per class (Training set)")
plt.xlabel("Class ID")
plt.ylabel("Number of images")
plt.show()

In [ ]:
widths = train_data['Width']
heights = train_data['Height']

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(widths, bins=20, color='skyblue')
plt.title("Width distribution")
plt.xlabel("Width (pixels)")

plt.subplot(1,2,2)
sns.histplot(heights, bins=20, color='salmon')
plt.title("Height distribution")
plt.xlabel("Height (pixels)")

plt.show()

In [ ]:
sample_images = X_train[:1500]
pixel_values = np.array([cv2.cvtColor((img*255).astype('uint8'), cv2.COLOR_RGB2GRAY).flatten()
                         for img in sample_images])
pixel_values = pixel_values.flatten()

plt.figure(figsize=(10,5))
sns.histplot(pixel_values, bins=50, color='purple')
plt.title("Pixel intensity distribution (grayscale)")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.show()

---------------------------------------------------------------------------------------------- Feature extraction 
 ---------------------------------------------------------------------------------------------

In [2]:
np.save("X_train_hog.npy", X_train_feat)
np.save("y_train_hog.npy", y_train_feat)
np.save("X_test_hog.npy", X_test_feat)
np.save("y_test_hog.npy", y_test_feat)

print("HOG features saved successfully!")

NameError: name 'X_train_feat' is not defined

## Loading the data

In [2]:
x_train_hog = np.load(r"E:\opencv\ML with Python\GTSRB-Traffic-Sign-Classification-main\GTSRB-Traffic-Sign-Classification-main\Data\X_train_hog.npy")
x_test_hog = np.load(r"E:\opencv\ML with Python\GTSRB-Traffic-Sign-Classification-main\GTSRB-Traffic-Sign-Classification-main\Data\X_test_hog.npy")
y_train_hog = np.load(r"E:\opencv\ML with Python\GTSRB-Traffic-Sign-Classification-main\GTSRB-Traffic-Sign-Classification-main\Data\y_train_hog.npy")
y_test_hog = np.load(r"E:\opencv\ML with Python\GTSRB-Traffic-Sign-Classification-main\GTSRB-Traffic-Sign-Classification-main\Data\y_test_hog.npy")

# KNN model

In [52]:
knn_model = KNeighborsClassifier()

## Selecting the best model hyperparameters

In [53]:
param_grid={
    "n_neighbors":[3,5,7],
    "weights":["uniform","distance"],
    'metric':['euclidean','manhattan']
}

In [54]:
x_train_GridSearch_sample , y_train_GridSearch_sample = x_train_hog[:15000] , y_train_hog[:15000]
grid = GridSearchCV(estimator=knn_model , param_grid=param_grid , scoring='accuracy')
grid.fit(x_train_GridSearch_sample, y_train_GridSearch_sample)
print(f"best hyperparameters for accuracy: {grid.best_params_}")

best hyperparameters for accuracy: {'metric': 'manhattan', 'n_neighbors': 7, 'weights': 'distance'}


## Training the model

In [55]:
knn_model = KNeighborsClassifier(n_neighbors=7 , metric='manhattan' , weights='uniform')
knn_model.fit(x_train_hog, y_train_hog)

,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'manhattan'
,metric_params,None
,n_jobs,None


## evaluating the model

In [56]:
y_predict = knn_model.predict(x_test_hog)

In [57]:
knn_accuracy_score = accuracy_score(y_test_hog,y_predict)
knn_precision_score = precision_score(y_test_hog,y_predict, average='macro')
knn_recall_score = recall_score(y_test_hog,y_predict, average='macro')
knn_f1Score = f1_score(y_test_hog,y_predict, average='macro')
knn_confusion_matrix = confusion_matrix(y_test_hog,y_predict)

In [58]:
print(f"accuracy: {knn_accuracy_score}")
print(f"precision score: {knn_precision_score}")
print(f"recall score: {knn_recall_score}")
print(f"f1 score: {knn_f1Score}")
print(f"confusion matrix:\n{knn_confusion_matrix}")

accuracy: 0.710055423594616
precision score: 0.7141390614199655
recall score: 0.6514753459662459
f1 score: 0.6707385978624811
confusion matrix:
[[ 13  16   8 ...   0   0   0]
 [  6 381 139 ...   0   0   0]
 [  0  80 444 ...   0   0   0]
 ...
 [  0   4   0 ...  64   0   0]
 [  0   1   1 ...   0  18   8]
 [  0   8   5 ...   0   0  56]]


# Naive Bayes model

In [45]:
naive_bayes_model = GaussianNB()

## Selecting the best model hyperparameters

In [46]:
param_grid={
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

In [47]:
x_train_GridSearch_sample, y_train_GridSearch_sample = x_train_hog[:39000] , y_train_hog[:39000]
grid = GridSearchCV(estimator=naive_bayes_model, param_grid=param_grid, scoring='accuracy')
grid.fit(x_train_GridSearch_sample, y_train_GridSearch_sample)
print(f"best hyperparameters for accuracy: {grid.best_params_}")

best hyperparameters for accuracy: {'var_smoothing': 1e-05}


## training the model

In [48]:
naive_bayes_model = GaussianNB(var_smoothing=1e-09)
naive_bayes_model.fit(x_train_hog, y_train_hog)

,priors,None
,var_smoothing,1e-09


## evaluating the model

In [49]:
y_predict = naive_bayes_model.predict(x_test_hog)

In [50]:
naive_bayes_accuracy_score = accuracy_score(y_test_hog,y_predict)
naive_bayes_precision_score = precision_score(y_test_hog,y_predict, average='macro')
naive_bayes_recall_score = recall_score(y_test_hog,y_predict, average='macro')
naive_bayes_f1Score = f1_score(y_test_hog,y_predict, average='macro')
naive_bayes_confusion_matrix = confusion_matrix(y_test_hog,y_predict)

In [51]:
print(f"accuracy: {naive_bayes_accuracy_score}")
print(f"precision score: {naive_bayes_precision_score}")
print(f"recall score: {naive_bayes_recall_score}")
print(f"f1 score: {naive_bayes_f1Score}")
print(f"confusion matrix:\n{naive_bayes_confusion_matrix}")

accuracy: 0.6601741884402217
precision score: 0.6504485669216327
recall score: 0.6597487908582449
f1 score: 0.637263528919588
confusion matrix:
[[ 33   4   3 ...   3   0   0]
 [ 77 306 107 ...   8   2   0]
 [ 23 126 336 ...  55   0   0]
 ...
 [  0   0   0 ...  76   0   0]
 [  0   3   0 ...   0  27   4]
 [  5   1   0 ...  12   0  55]]


## PCA for compression

In [59]:
print("Shape of X_train_hog:", x_train_hog.shape)
print("Number of features:", x_train_hog.shape[1])

Shape of X_train_hog: (39209, 324)
Number of features: 324


In [60]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)  # 95% important information
X_train_pca = pca.fit_transform(x_train_hog)
X_test_pca  = pca.transform(x_test_hog)

print("Number of features before PCA:", x_train_hog.shape[1])
print("Number of features after PCA:", X_train_pca.shape[1])

Number of features before PCA: 324
Number of features after PCA: 113


In [61]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score


# --- Train KNN with pca ---
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_pca, y_train_hog)
y_pred_knn = knn.predict(X_test_pca)
print("KNN accuracy after PCA:", accuracy_score(y_test_hog, y_pred_knn))

# --- Train Naive Bayes with pca ---
naive = GaussianNB()
naive.fit(X_train_pca, y_train_hog)
y_pred_naive = naive.predict(X_test_pca)
print("Naive Bayes accuracy after PCA:", accuracy_score(y_test_hog, y_pred_naive))

KNN accuracy after PCA: 0.6868566904196358
Naive Bayes accuracy after PCA: 0.7097387173396674


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

pca = PCA()
pca.fit(x_train_hog)

explained_variance = pca.explained_variance_ratio_

# cumulative explained variance
cumulative_variance = np.cumsum(explained_variance)

# Plot
plt.figure(figsize=(10,5))
plt.plot(cumulative_variance, marker='o', linestyle='--')
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("Explained Variance vs Number of PCA components")
plt.grid(True)
plt.show()

## Ensemble Learning

In [4]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    random_state=42
)

rf.fit(x_train_hog, y_train_hog)

y_pred_rf = rf.predict(x_test_hog)

print("Random Forest Accuracy:", accuracy_score(y_test_hog, y_pred_rf))

Random Forest Accuracy: 0.769200316706255


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3
)

gb.fit(x_train_hog, y_train_hog)

y_pred_gb = gb.predict(x_test_hog)

print("Gradient Boosting Accuracy:", accuracy_score(y_test_hog, y_pred_gb))

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('nb', GaussianNB()),
        ('rf', RandomForestClassifier())
    ],
    voting='hard'
)

ensemble.fit(x_train_hog, y_train_hog)

y_pred = ensemble.predict(x_test_hog)

print("Ensemble Accuracy:", accuracy_score(y_test_hog, y_pred))